# Домашнее задание 7. Работа с API

Запустите нужную ячейку сочетанием **Shift + Enter**. Для запросов нужен интернет. Достаточно стандартной библиотеки Python, ключ API не требуется.


## Задание 1. Первые пять постов

Отправляем GET-запрос к `/posts` API JSONPlaceholder. Из JSON-ответа берём первые пять записей и печатаем поля `title` и `body`.


In [ ]:
import json
from urllib.error import URLError
from urllib.request import urlopen

POSTS_URL = "https://jsonplaceholder.typicode.com/posts"


def main():
    try:
        with urlopen(POSTS_URL, timeout=10) as response:
            posts = json.load(response)
    except (URLError, TimeoutError, ValueError) as error:
        print(f"Не удалось получить посты: {error}")
        return

    for number, post in enumerate(posts[:5], start=1):
        print(f"{number}. {post['title']}")
        print(post["body"])
        print()

main()


## Задание 2. Текущая погода

Запустите ячейку и введите название города, например **Москва**. Первый GET-запрос передаёт название в Open-Meteo Geocoding API. Второй передаёт координаты первого найденного города в Open-Meteo Forecast API и запрашивает текущую температуру и код погоды. Программа выводит температуру в градусах Цельсия и описание на русском языке.


In [ ]:
import json
from urllib.error import URLError
from urllib.parse import urlencode
from urllib.request import urlopen

GEOCODING_URL = "https://geocoding-api.open-meteo.com/v1/search"
FORECAST_URL = "https://api.open-meteo.com/v1/forecast"

WEATHER_DESCRIPTIONS = {
    0: "Ясно",
    1: "Преимущественно ясно",
    2: "Переменная облачность",
    3: "Пасмурно",
    45: "Туман",
    48: "Туман с изморозью",
    51: "Слабая морось",
    53: "Умеренная морось",
    55: "Сильная морось",
    56: "Слабая ледяная морось",
    57: "Сильная ледяная морось",
    61: "Слабый дождь",
    63: "Умеренный дождь",
    65: "Сильный дождь",
    66: "Слабый ледяной дождь",
    67: "Сильный ледяной дождь",
    71: "Слабый снег",
    73: "Умеренный снег",
    75: "Сильный снег",
    77: "Снежные зёрна",
    80: "Слабый ливень",
    81: "Умеренный ливень",
    82: "Сильный ливень",
    85: "Слабый снежный ливень",
    86: "Сильный снежный ливень",
    95: "Гроза",
    96: "Гроза со слабым градом",
    97: "Сильная гроза",
    99: "Гроза с сильным градом",
}


def get_json(url, params):
    query = urlencode(params)
    with urlopen(f"{url}?{query}", timeout=10) as response:
        return json.load(response)


def main():
    city = input("Введите название города: ").strip()
    if not city:
        print("Название города не может быть пустым.")
        return

    try:
        places = get_json(GEOCODING_URL, {"name": city, "count": 1, "language": "ru"})
        results = places.get("results", [])
        if not results:
            print(f"Город «{city}» не найден.")
            return

        place = results[0]
        forecast = get_json(FORECAST_URL, {
            "latitude": place["latitude"],
            "longitude": place["longitude"],
            "current": "temperature_2m,weather_code",
            "temperature_unit": "celsius",
        })
        current = forecast["current"]
        temperature = current["temperature_2m"]
        code = current["weather_code"]
    except (URLError, TimeoutError, ValueError, KeyError) as error:
        print(f"Не удалось получить погоду: {error}")
        return

    if temperature is None or code is None:
        print("Текущие данные о погоде недоступны.")
        return

    location = ", ".join(part for part in (place["name"], place.get("country")) if part)
    description = WEATHER_DESCRIPTIONS.get(code, f"Неизвестный код погоды ({code})")
    print(f"Город: {location}")
    print(f"Текущая температура: {temperature} °C")
    print(f"Погода: {description}")

main()
